# Near-Duplicate Review and Dataset Leakage Analysis

This notebook reviews near-duplicate images detected across the
train, validation, and test splits of the Domestic Animal Classification
dataset.

## Objective

The goal is to identify potential cross-split image leakage while
preserving as much valid training data as possible.

The workflow is:

1. Load pHash-based near-duplicate results.
2. Focus on the highest-confidence pHash candidates.
3. Verify candidates using ORB feature matching.
4. Validate the ORB matching threshold using representative examples.
5. Group connected duplicate candidates.
6. Preserve the highest-priority dataset split.
7. Generate a cleanup recommendation report.

No files are deleted automatically in this notebook.

In [1]:
from pathlib import Path
from PIL import Image
from collections import Counter

import cv2
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Paths
RESULTS_DIR = Path("../results")

CROSS_SPLIT_FILE = RESULTS_DIR / "cross_split_near_duplicates.csv"
CLEANUP_REPORT_FILE = RESULTS_DIR / "final_duplicate_cleanup_report.csv"

In [4]:
# Duplicate-detection configuration
PHASH_DISTANCE = 0
ORB_INLIER_THRESHOLD = 11

# Dataset split priority
# Higher priority means the image is preferred for retention.
SPLIT_PRIORITY = {
    "validation": 1,
    "test": 2,
    "train": 3,
}

## 1. Load Near-Duplicate Results

The previous notebook (`03_check_near_duplicates.ipynb`) detected
near-duplicate image pairs using perceptual hashing (pHash).

This notebook uses the saved cross-split results rather than
re-running the pHash search.

In [5]:
# Make sure the previous notebook has produced the required results file.
if not CROSS_SPLIT_FILE.exists():
    raise FileNotFoundError(
        f"Required file not found: {CROSS_SPLIT_FILE}"
    )

review_df = pd.read_csv(CROSS_SPLIT_FILE)

print(f"Cross-split near-duplicate pairs: {len(review_df)}")

Cross-split near-duplicate pairs: 382


In [7]:
# Summarize the pHash distance distribution.
distance_counts = (
    review_df["distance"]
    .value_counts()
    .sort_index()
)

print("pHash distance distribution:")
display(distance_counts)

pHash distance distribution:


distance
0    333
2     44
4      5
Name: count, dtype: int64

## 2. Select High-Confidence pHash Candidates

A pHash distance of 0 indicates that two images have identical
perceptual hashes.

These pairs are selected for additional verification using ORB.

In [8]:
# Select cross-split pairs with identical perceptual hashes.
high_confidence = review_df[
    review_df["distance"] == PHASH_DISTANCE
].copy()

print(
    f"pHash-distance-{PHASH_DISTANCE} candidates: "
    f"{len(high_confidence)}"
)

pHash-distance-0 candidates: 333


In [9]:
# Show the candidate distribution across dataset classes.
print("Candidates by class:")
display(high_confidence["class1"].value_counts())

Candidates by class:


class1
Chicken    91
Donkey     87
Horse      64
Goat       29
Sheep      28
Camel      26
Pig         8
Name: count, dtype: int64

## 3. Verify Candidates Using ORB

Perceptual hashing is useful for finding visually similar images,
but it is not sufficient by itself to determine whether two files
represent the same underlying image.

ORB feature matching is therefore used as a second verification step.

Two values are recorded:

- Good matches: feature correspondences that pass Lowe's ratio test.
- Inliers: matches that remain geometrically consistent after RANSAC.

The current experimental threshold is 11 ORB inliers.

In [10]:
def orb_match_score(image_path_1, image_path_2):
    """
    Compare two images using ORB feature matching.

    Returns:
        good_matches: Number of reliable feature matches.
        inliers: Number of geometrically consistent matches.
    """

    image_1 = cv2.imread(str(image_path_1), cv2.IMREAD_GRAYSCALE)
    image_2 = cv2.imread(str(image_path_2), cv2.IMREAD_GRAYSCALE)

    if image_1 is None or image_2 is None:
        return 0, 0

    orb = cv2.ORB_create(nfeatures=1000)

    keypoints_1, descriptors_1 = orb.detectAndCompute(image_1, None)
    keypoints_2, descriptors_2 = orb.detectAndCompute(image_2, None)

    if descriptors_1 is None or descriptors_2 is None:
        return 0, 0

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING)

    matches = matcher.knnMatch(
        descriptors_1,
        descriptors_2,
        k=2
    )

    # Keep matches that pass Lowe's ratio test.
    good_matches = []

    for match_pair in matches:
        if len(match_pair) == 2:
            match_1, match_2 = match_pair

            if match_1.distance < 0.75 * match_2.distance:
                good_matches.append(match_1)

    if len(good_matches) < 4:
        return len(good_matches), 0

    points_1 = np.float32(
        [keypoints_1[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    points_2 = np.float32(
        [keypoints_2[m.trainIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    _, mask = cv2.findHomography(
        points_1,
        points_2,
        cv2.RANSAC,
        5.0
    )

    inliers = int(mask.sum()) if mask is not None else 0

    return len(good_matches), inliers

In [11]:
# Verify every pHash-distance-0 candidate using ORB.
orb_results = []

for _, row in high_confidence.iterrows():

    good_matches, inliers = orb_match_score(
        row["image1"],
        row["image2"]
    )

    orb_results.append({
        "image1": row["image1"],
        "split1": row["split1"],
        "class1": row["class1"],
        "image2": row["image2"],
        "split2": row["split2"],
        "class2": row["class2"],
        "phash_distance": row["distance"],
        "good_matches": good_matches,
        "inliers": inliers,
    })

orb_df = pd.DataFrame(orb_results)

print(f"ORB verification completed: {len(orb_df)} pairs")

ORB verification completed: 333 pairs


## 4. Validate the ORB Threshold

The initial pHash-distance-0 set contained 333 cross-split pairs.

Several representative pairs near the lower end of the ORB inlier
range were visually inspected.

The inspected examples had 11, 16, 20, 26, and 47 inliers, and all
five inspected pairs represented the same underlying image.

Based on this validation, 11 inliers is used as the current working
threshold for automated cleanup candidate selection.

This threshold is treated as an empirical project rule rather than
a universal ORB threshold.

In [12]:
high_confidence = review_df[
    review_df["distance"] == 0
].copy()

print(f"High-confidence cross-split pairs: {len(high_confidence)}")

print("\nDistribution by class:")
print(high_confidence["class1"].value_counts())

print("\nDistribution by split combination:")
print(
    high_confidence[
        ["split1", "split2"]
    ].value_counts()
)

High-confidence cross-split pairs: 333

Distribution by class:
class1
Chicken    91
Donkey     87
Horse      64
Goat       29
Sheep      28
Camel      26
Pig         8
Name: count, dtype: int64

Distribution by split combination:
split1      split2    
train       test          170
            validation    104
validation  test           59
Name: count, dtype: int64


## 5. Apply the Validated ORB Threshold

The validated working threshold is **11 ORB inliers**.

A pair is treated as a confirmed duplicate candidate when:

- pHash distance = 0
- ORB inliers >= 11

This produces the set of candidate relationships that will be used for
duplicate-group analysis.

The threshold is specific to this dataset and should not be interpreted
as a universal ORB duplicate-detection threshold.

In [13]:
# Select candidates that satisfy the validated pHash and ORB criteria.
confirmed_candidates = orb_df[
    (orb_df["phash_distance"] == PHASH_DISTANCE) &
    (orb_df["inliers"] >= ORB_INLIER_THRESHOLD)
].copy()

print(
    f"Confirmed duplicate candidates: "
    f"{len(confirmed_candidates)}"
)

print("\nCandidates by split combination:")
display(
    confirmed_candidates[
        ["split1", "split2"]
    ].value_counts()
)

Confirmed duplicate candidates: 293

Candidates by split combination:


split1      split2    
train       test          148
            validation     93
validation  test           52
Name: count, dtype: int64

In [14]:
# Summarize confirmed candidates by class.
print("Confirmed candidates by class:")
display(
    confirmed_candidates["class1"]
    .value_counts()
)

print("\nConfirmed candidates by split combination:")
display(
    confirmed_candidates[
        ["split1", "split2"]
    ]
    .value_counts()
)

Confirmed candidates by class:


class1
Chicken    83
Donkey     71
Horse      63
Goat       28
Camel      22
Sheep      22
Pig         4
Name: count, dtype: int64


Confirmed candidates by split combination:


split1      split2    
train       test          148
            validation     93
validation  test           52
Name: count, dtype: int64

## 7. Build Connected Duplicate Groups

Duplicate detection produces relationships between pairs of images.

However, multiple pairwise relationships may belong to the same family
of duplicated images.

For example:

Image A ↔ Image B  
Image B ↔ Image C

These three images should be treated as one duplicate group.

A graph is therefore created where:

- each image is a node
- each confirmed duplicate relationship is an edge
- each connected component represents one duplicate group

This prevents cleanup decisions from being made independently for each
pair.

In [15]:
# Create a graph of confirmed duplicate relationships.
graph = nx.Graph()

for _, row in confirmed_candidates.iterrows():

    image1 = row["image1"]
    image2 = row["image2"]

    # Connect the two related images.
    graph.add_edge(image1, image2)

# Extract connected components as duplicate groups.
duplicate_groups = list(
    nx.connected_components(graph)
)

print(
    f"Duplicate groups found: "
    f"{len(duplicate_groups)}"
)

# Show the largest groups to understand the structure.
group_sizes = sorted(
    [len(group) for group in duplicate_groups],
    reverse=True
)

print("\nLargest duplicate groups:")
print(group_sizes[:20])

Duplicate groups found: 264

Largest duplicate groups:
[6, 5, 5, 4, 4, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2]


## 8. Analyze Duplicate Group Composition

Before deciding which images to retain, we examine how the duplicate
groups are distributed across the dataset splits.

This is important because a group may contain:

- one image in Train and one in Test
- one image in Train and one in Validation
- multiple images in Train and one in Test
- multiple images within the same split

The split composition determines which copy should be preserved.

In [16]:
from collections import Counter


def get_split(image_path):
    """
    Return the dataset split for an image path.

    Expected structure:
        data/
        ├── train/
        ├── validation/
        └── test/
    """

    path = Path(image_path)

    for split in ["train", "validation", "test"]:
        if split in path.parts:
            return split

    return "unknown"

In [17]:
# Summarize how many images from each split occur in every group.
group_summary = []

for group_number, group in enumerate(
    duplicate_groups,
    start=1
):

    # Identify the split of every image in the group.
    splits = [
        get_split(image_path)
        for image_path in group
    ]

    split_counts = Counter(splits)

    group_summary.append({
        "group": group_number,
        "group_size": len(group),
        "train": split_counts.get("train", 0),
        "validation": split_counts.get("validation", 0),
        "test": split_counts.get("test", 0),
    })

group_summary_df = pd.DataFrame(group_summary)

print("Duplicate groups by split composition:")
display(
    group_summary_df[
        ["train", "validation", "test"]
    ]
    .value_counts()
    .sort_index()
)

Duplicate groups by split composition:


train  validation  test
0      1           1        49
1      0           1       130
                   2         1
       1           0        73
       2           0         1
2      0           1         2
       1           0         3
                   1         1
       2           1         1
3      0           1         1
       2           0         1
5      0           1         1
Name: count, dtype: int64

## 10. Identify Groups Requiring Special Handling

Most duplicate groups contain one image in the highest-priority split
and one or more images in lower-priority splits.

A small number of groups contain multiple images in the highest-priority
split itself.

These groups cannot be resolved using split priority alone, so they are
examined separately before the final cleanup report is created.

In [18]:
# Define the priority of each dataset split.
# Higher priority means the split is preferred for retention.
SPLIT_PRIORITY = {
    "train": 1,
    "validation": 2,
    "test": 3,
}


def get_highest_priority_split(group):
    """Return the highest-priority split represented in a group."""

    splits = [
        get_split(image_path)
        for image_path in group
    ]

    return max(
        splits,
        key=lambda split: SPLIT_PRIORITY[split]
    )


# Identify groups containing multiple images in the preferred split.
special_groups = []

for group_number, group in enumerate(
    duplicate_groups,
    start=1
):

    keep_split = get_highest_priority_split(group)

    keep_count = sum(
        get_split(image_path) == keep_split
        for image_path in group
    )

    if keep_count > 1:
        special_groups.append({
            "group": group_number,
            "group_size": len(group),
            "keep_split": keep_split,
            "keep_count": keep_count,
        })

special_groups_df = pd.DataFrame(
    special_groups
)

print(
    f"Groups with multiple images in the "
    f"highest-priority split: {len(special_groups_df)}"
)

display(special_groups_df)

Groups with multiple images in the highest-priority split: 3


,group,group_size,keep_split,keep_count
0,78,5,validation,2
1,90,3,test,2
2,94,3,validation,2


## 11. Review Special Duplicate Groups

Three groups were found to contain multiple images in the preferred
dataset split:

- Group 78 — Validation
- Group 90 — Test
- Group 94 — Validation

These groups were inspected using image dimensions and file size.

The inspection confirmed that the images within each group represent the
same underlying image at different resolutions or encodings.

For these cases, the highest-resolution image within the highest-priority
split is retained.

In [19]:
# Inspect the images in the special groups.
# This is a read-only inspection; no files are changed.

for group_number in special_groups_df["group"]:

    group = duplicate_groups[group_number - 1]

    print("=" * 70)
    print(f"GROUP {group_number} — {len(group)} images")
    print("=" * 70)

    for image_path in group:

        path = Path(image_path)
        split = get_split(path)

        # The class is the directory immediately above the image file.
        class_name = path.parent.name

        try:
            # Read image dimensions and file size.
            with Image.open(path) as image:
                width, height = image.size

            file_size_kb = path.stat().st_size / 1024

            print(
                f"{split:12} | "
                f"{class_name:10} | "
                f"{width}x{height:<5} | "
                f"{file_size_kb:8.1f} KB | "
                f"{path.name}"
            )

        except Exception as error:

            print(
                f"{split:12} | "
                f"{class_name:10} | "
                f"Could not inspect file | "
                f"{path.name}"
            )

            print(f"Error: {error}")

    print()

GROUP 78 — 5 images
validation   | Donkey     | 1000x668   |    246.8 KB | 1000_F_117351782_ugRMzKUg8pz8ucKVqIPI1JzQSCItC0Hx.jpg
train        | Donkey     | 500x334   |     80.6 KB | 500_F_117351782_ugRMzKUg8pz8ucKVqIPI1JzQSCItC0Hx.jpg
validation   | Donkey     | 360x240   |     32.8 KB | 240_F_117351782_ugRMzKUg8pz8ucKVqIPI1JzQSCItC0Hx.jpg
train        | Donkey     | 400x267   |     22.3 KB | funny-donkey-road-transfagarasan-romanian-mountains-75232844 (5).jpg
train        | Donkey     | 539x360   |     51.1 KB | 360_F_117351782_ugRMzKUg8pz8ucKVqIPI1JzQSCItC0Hx.jpg

GROUP 90 — 3 images
test         | Donkey     | 361x240   |     42.4 KB | 176786131.jpg
train        | Donkey     | 800x533   |     63.5 KB | donkey-funny-animals-donkey-funny-animals-happy-humorous-donkey-laughing-something-very-very-funny-101898764 (1).jpg
test         | Donkey     | 541x360   |     63.1 KB | 360_F_176786131_KnCinzfMZUaMLeQMKcMoYrBw3UGluQ15.jpg

GROUP 94 — 3 images
validation   | Donkey     | 479x360   |

In [20]:
# Build split-aware cleanup recommendations.
# No files are deleted or moved by this code.

final_cleanup = []

for group_number, group in enumerate(
    duplicate_groups,
    start=1
):

    group_images = list(group)

    image_info = []

    for image_path in group_images:

        path = Path(image_path)
        split = get_split(path)

        try:
            # Read image dimensions for quality comparison.
            with Image.open(path) as image:
                width, height = image.size

            resolution = (width or 0) * (height or 0)

        except Exception:
            width = None
            height = None
            resolution = 0

        image_info.append({
            "path": str(path),
            "split": split,
            "width": width,
            "height": height,
            "resolution": resolution,
            "file_size_kb": round(
                path.stat().st_size / 1024,
                1
            ),
        })

    # Group images by dataset split.
    images_by_split = {}

    for item in image_info:
        images_by_split.setdefault(
            item["split"],
            []
        ).append(item)

    # ---------------------------------------------------------
    # Determine which split should be retained.
    #
    # Priority:
    # Train > Test > Validation
    # ---------------------------------------------------------

    if "train" in images_by_split:
        preferred_split = "train"

    elif "test" in images_by_split:
        preferred_split = "test"

    elif "validation" in images_by_split:
        preferred_split = "validation"

    else:
        preferred_split = None

    if preferred_split is None:
        continue

    # ---------------------------------------------------------
    # Keep the highest-resolution image from the preferred split.
    # ---------------------------------------------------------

    preferred_images = images_by_split[preferred_split]

    keep_image = max(
        preferred_images,
        key=lambda item: (
            item["resolution"],
            item["file_size_kb"]
        )
    )

    # ---------------------------------------------------------
    # Determine which images should be removed.
    # ---------------------------------------------------------

    remove_images = []

    for item in image_info:

        # Never remove the selected keep image.
        if item["path"] == keep_image["path"]:
            continue

        # If another image is from the same preferred split,
        # remove it only when it is an extra duplicate.
        if item["split"] == preferred_split:
            remove_images.append(item)
            continue

        # Train is preferred over validation and test.
        if preferred_split == "train":
            remove_images.append(item)
            continue

        # Test is preferred over validation.
        if (
            preferred_split == "test"
            and item["split"] == "validation"
        ):
            remove_images.append(item)
            continue

    # ---------------------------------------------------------
    # Create cleanup records.
    # ---------------------------------------------------------

    for remove_image in remove_images:

        final_cleanup.append({
            "group": group_number,

            "keep_split": keep_image["split"],
            "keep_image": keep_image["path"],
            "keep_resolution": (
                f"{keep_image['width']}x"
                f"{keep_image['height']}"
            ),

            "remove_split": remove_image["split"],
            "remove_image": remove_image["path"],
            "remove_resolution": (
                f"{remove_image['width']}x"
                f"{remove_image['height']}"
            ),

            "reason": (
                "Duplicate group; retain the preferred dataset split "
                "(Train > Test > Validation) and the highest-resolution "
                "copy within that split."
            ),
        })


# Convert recommendations into a DataFrame.
final_cleanup_df = pd.DataFrame(
    final_cleanup
)

print(
    f"Duplicate groups analyzed: "
    f"{len(duplicate_groups)}"
)

print(
    f"Images recommended for removal: "
    f"{len(final_cleanup_df)}"
)

if not final_cleanup_df.empty:
    print("\nCandidates by split combination:")

    print(
        final_cleanup_df[
            ["keep_split", "remove_split"]
        ]
        .value_counts()
    )

Duplicate groups analyzed: 264
Images recommended for removal: 285

Candidates by split combination:
keep_split  remove_split
train       test            138
            validation       83
test        validation       49
train       train            15
Name: count, dtype: int64


In [21]:
# Summarize proposed removals by dataset split.
print("Recommended removals by split:")

display(
    final_cleanup_df["remove_split"]
    .value_counts()
    .rename_axis("split")
    .to_frame("removal_count")
)

print("\nFirst 10 recommendations:")

display(
    final_cleanup_df.head(10)
)

Recommended removals by split:


,removal_count
split,
test,138
validation,132
train,15



First 10 recommendations:


,group,keep_split,keep_image,keep_resolution,remove_split,remove_image,remove_resolution,reason
0,1,train,..\data\train\Camel\Free Shallow Focus Photogr...,500x333,test,..\data\test\Camel\5M0K2I2A2BSZ.jpg,256x256,Duplicate group; retain the preferred dataset ...
1,2,train,..\data\train\Camel\G4PZDEMX453U.jpg,256x256,test,..\data\test\Camel\1c68ec73964a0332.jpg,1024x681,Duplicate group; retain the preferred dataset ...
2,3,train,..\data\train\Camel\GUDRW05GNVCD.jpg,256x256,test,..\data\test\Camel\4bc1027f1ccb353d.jpg,1024x683,Duplicate group; retain the preferred dataset ...
3,4,train,..\data\train\Camel\happy-camel.jpg,612x408,test,..\data\test\Camel\415AZP8HOFL9.jpg,256x256,Duplicate group; retain the preferred dataset ...
4,5,train,..\data\train\Camel\HZEFVYXLAYKJ.jpg,256x256,test,..\data\test\Camel\camel_newborn_dromedary.jpg,1000x667,Duplicate group; retain the preferred dataset ...
5,6,train,..\data\train\Camel\IK3X3JEKGWH6.jpg,256x256,test,..\data\test\Camel\9bb8fd51ad6846e9.jpg,1024x1024,Duplicate group; retain the preferred dataset ...
6,7,train,..\data\train\Camel\J3JLAJDY974S.jpg,256x256,test,..\data\test\Camel\0a5e8d559aa5d680.jpg,1024x819,Duplicate group; retain the preferred dataset ...
7,8,train,..\data\train\Camel\JSC59TXM8HBO.jpg,256x256,test,..\data\test\Camel\9219e0b5c16a76dc.jpg,1024x768,Duplicate group; retain the preferred dataset ...
8,9,train,..\data\train\Camel\LH9SC7Z1ZGKZ.jpg,256x256,test,..\data\test\Camel\0b561c9610d097bb.jpg,1024x680,Duplicate group; retain the preferred dataset ...
9,10,train,..\data\train\Camel\M7DUD6DEUZXC.jpg,256x256,test,..\data\test\Camel\fbe95350ddcee63a.jpg,1024x768,Duplicate group; retain the preferred dataset ...


## 12. Save the Cleanup Report

The final recommendations are saved as a CSV file.

This file provides a reproducible record of:

- the duplicate group
- the image being retained
- the image being removed
- the dataset split of each image
- the resolution of each image
- the reason for the recommendation

The report can be reviewed independently before any dataset files are
modified.

In [22]:
# Save the final cleanup recommendations.
CLEANUP_REPORT_FILE = (
    RESULTS_DIR /
    "final_duplicate_cleanup_report.csv"
)

final_cleanup_df.to_csv(
    CLEANUP_REPORT_FILE,
    index=False
)

print("Cleanup report saved to:")
print(CLEANUP_REPORT_FILE)

Cleanup report saved to:
..\results\final_duplicate_cleanup_report.csv


## 15. Quarantine Recommended Duplicate Files

In [23]:
import shutil

# Paths
DATASET_ROOT = Path("../data").resolve()
QUARANTINE_ROOT = Path("../quarantine").resolve()
REPORT_FILE = Path("../results/final_duplicate_cleanup_report.csv").resolve()

# Load the final cleanup report
cleanup_df = pd.read_csv(REPORT_FILE)

print(f"Cleanup recommendations found: {len(cleanup_df)}")

# Create quarantine directory
QUARANTINE_ROOT.mkdir(parents=True, exist_ok=True)

quarantine_log = []

for _, row in cleanup_df.iterrows():

    source = Path(row["remove_image"])

    try:
        # Convert relative paths into absolute paths
        source = source.resolve()

        # Make sure the file is actually inside the dataset
        relative_path = source.relative_to(DATASET_ROOT)

        # Preserve the original dataset folder structure
        destination = QUARANTINE_ROOT / relative_path

        # Check that the source file exists
        if not source.exists():
            quarantine_log.append({
                "status": "MISSING",
                "source": str(source),
                "destination": str(destination),
                "reason": "Source file does not exist"
            })
            continue

        # Create destination folder
        destination.parent.mkdir(parents=True, exist_ok=True)

        # Move the file to quarantine
        shutil.move(str(source), str(destination))

        quarantine_log.append({
            "status": "MOVED",
            "source": str(source),
            "destination": str(destination),
            "reason": row["reason"]
        })

    except Exception as e:
        quarantine_log.append({
            "status": "ERROR",
            "source": str(source),
            "destination": "",
            "reason": str(e)
        })

# Save a log of everything that happened
quarantine_log_df = pd.DataFrame(quarantine_log)

log_file = Path("../results/quarantine_log.csv")
quarantine_log_df.to_csv(log_file, index=False)

# Summary
print("\nQuarantine completed.")
print(f"Moved:    {(quarantine_log_df['status'] == 'MOVED').sum()}")
print(f"Missing:  {(quarantine_log_df['status'] == 'MISSING').sum()}")
print(f"Errors:   {(quarantine_log_df['status'] == 'ERROR').sum()}")

print(f"\nQuarantine folder: {QUARANTINE_ROOT}")
print(f"Log saved to: {log_file.resolve()}")

Cleanup recommendations found: 285

Quarantine completed.
Moved:    285
Missing:  0
Errors:   0

Quarantine folder: D:\Domestic Animal Classification\quarantine
Log saved to: D:\Domestic Animal Classification\results\quarantine_log.csv
